# Hyperscale Data Center — Failure Risk Prediction

**Dataset:** [Hyperscale Data Center Dataset (Kaggle)](https://www.kaggle.com/datasets/deeplumiere/hyperscale-data-center-dataset/)

## Problem

We have ~100,000 datacenter telemetry records (CPU/GPU load, cooling, energy, region, etc.).
The chosen target is **`failure_risk_level`** (ordinal classes: `Low` / `Medium` / `High`).

Goals:
- build a classifier that is robust to **class imbalance** (~49% Low, ~34% Medium, ~16% High);
- use a metric that is not inflated by the majority class: **`balanced_accuracy`**;
- compare models suited to **mixed features** (categorical + numeric) and potentially **non-linear** relationships.


## 1. Imports & paths


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split, RandomizedSearchCV, PredefinedSplit
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgbm


In [ ]:
def get_data_path() -> Path:
    kaggle_path = Path("/kaggle/input/datasets/deeplumiere/hyperscale-data-center-dataset/")
    if kaggle_path.exists():
        return kaggle_path
    return Path(".")  
DATA_PATH = get_data_path()


## 2. Data loading

### Watch out: `datacenter_region` and the `NA` value

**North America** is encoded as `NA` in the CSV.
By default, `pandas.read_csv` treats `"NA"` as a missing value (`NaN`), which created ~28.8% *fake* missing values.

**Fix:** `keep_default_na=False` + `na_values=[""]` so only truly empty cells are treated as missing.
This keeps `NA` as a valid category (`EU`, `APAC`, `ME`, `NA`).


In [ ]:
df = pd.read_csv(DATA_PATH / "green_ai_datacenter.csv", keep_default_na=False, na_values=[""])
df.head()


## 3. Quick EDA

Check the schema, numeric distributions, and confirm there are no real missing values after fixing the load.


In [ ]:
df.shape


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
(df.isnull().sum() / len(df)).sort_values(ascending=False)


### Regions & target

After fixing the parser, `datacenter_region` should no longer contain NaNs.
We also check the `failure_risk_level` distribution (imbalance to account for in the model and the metric).


In [ ]:
print(df["datacenter_region"].value_counts(dropna=False))
print()
print(df["failure_risk_level"].value_counts(dropna=False))


## 4. Modeling — `failure_risk_level`

### Model choices

The dataset combines:
- **categorical** variables (region, server type, energy source, etc.);
- continuous **numeric** variables (utilizations, temperatures, PUE, carbon intensity, ...);
- a **large volume** (~100k rows);
- likely **non-linear** interactions.

In this setting, **gradient boosting** (LightGBM, XGBoost) is a natural fit: strong on mixed tabular data, fast with histogram mode (`hist`), and able to handle imbalance (`class_weight` / `sample_weight`).

Random Forest was dropped for hyperparameter search: too slow on 100k rows × randomized search.

### Preprocessing

- `LabelEncoder` on the target (convenient / required for XGBoost with sklearn).
- `OrdinalEncoder` on categoricals only — a single transformer, no `Pipeline` (intentionally kept simple).
- Stratified train/test split, then an inner tune/val split + `PredefinedSplit` for `RandomizedSearchCV` without touching the hold-out test set.

### Metric

**`balanced_accuracy`**: mean of per-class recalls, suited to imbalanced multi-class problems.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns="failure_risk_level"), df['failure_risk_level'], stratify=df['failure_risk_level'], random_state=42)


In [ ]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)
print("Label mapping:", dict(enumerate(label_encoder.classes_)))

# Single categorical transformer: OrdinalEncoder is enough here (no Pipeline).
ordinal_encoder = OrdinalEncoder()
cat_cols = X_train.select_dtypes(include="object").columns.to_list()
X_train[cat_cols] = ordinal_encoder.fit_transform(X_train[cat_cols])
X_test[cat_cols] = ordinal_encoder.transform(X_test[cat_cols])

# Inner split for tuning (keeps the hold-out test set untouched).
X_tune, X_val, y_tune, y_val = train_test_split(
    X_train, y_train_encoded, test_size=0.15, random_state=42, stratify=y_train_encoded
)
X_search = pd.concat([X_tune, X_val])
y_search = np.concatenate([y_tune, y_val])
split = PredefinedSplit(test_fold=[-1] * len(X_tune) + [0] * len(X_val))


### 4.1 LightGBM

Main model: fast on large data, native `class_weight="balanced"`, limited random search (`n_iter=20`) with a predefined validation fold.


In [ ]:
lgbm_search = RandomizedSearchCV(
    lgbm.LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1),
    param_distributions={
        "n_estimators": [300, 500, 800, 1000],
        "max_depth": [4, 6, 8, 10, None],
        "learning_rate": [0.03, 0.05, 0.1],
        "num_leaves": [15, 31, 63, 127],
        "subsample": [0.7, 0.8, 1.0],
        "colsample_bytree": [0.7, 0.8, 1.0],
        "min_child_samples": [5, 10, 20, 50],
        "lambda_l2": [0.1, 1.0, 5.0],
    },
    n_iter=20,
    cv=split,
    scoring="balanced_accuracy",
    random_state=42,
    n_jobs=-1,
)

lgbm_search.fit(X_search, y_search)
print(f"Best CV balanced accuracy: {lgbm_search.best_score_:.4f}")
print(f"Best params: {lgbm_search.best_params_}")

lgbm_best_estimator = lgbm_search.best_estimator_
y_pred_lgbm = lgbm_best_estimator.predict(X_test)
print(f"Test balanced accuracy: {balanced_accuracy_score(y_test_encoded, y_pred_lgbm):.4f}")

lgbm_cm = confusion_matrix(y_test_encoded, y_pred_lgbm)
plt.figure(figsize=(8, 6))
sns.heatmap(
    lgbm_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("LightGBM — Confusion Matrix")
plt.tight_layout()
plt.show()


### 4.2 XGBoost

Comparison with XGBoost (`tree_method="hist"` for speed).
Note: XGBoost has **no** `class_weight` parameter — we use `compute_sample_weight("balanced")` passed to `fit`.


In [ ]:
sample_weight = compute_sample_weight("balanced", y_search)

xgb_search = RandomizedSearchCV(
    XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    ),
    param_distributions={
        "n_estimators": [300, 500, 800, 1000],
        "max_depth": [4, 6, 8, 10],
        "learning_rate": [0.03, 0.05, 0.1],
        "subsample": [0.7, 0.8, 1.0],
        "colsample_bytree": [0.7, 0.8, 1.0],
        "min_child_weight": [1, 3, 5, 10],
        "reg_lambda": [0.1, 1.0, 5.0],
        "gamma": [0, 0.1, 1.0],
    },
    n_iter=20,
    cv=split,
    scoring="balanced_accuracy",
    random_state=42,
    n_jobs=-1,
)

xgb_search.fit(X_search, y_search, sample_weight=sample_weight)
print(f"Best CV balanced accuracy: {xgb_search.best_score_:.4f}")
print(f"Best params: {xgb_search.best_params_}")

xgb_best_estimator = xgb_search.best_estimator_
y_pred_xgb = xgb_best_estimator.predict(X_test)
print(f"Test balanced accuracy: {balanced_accuracy_score(y_test_encoded, y_pred_xgb):.4f}")

xgb_cm = confusion_matrix(y_test_encoded, y_pred_xgb)
plt.figure(figsize=(8, 6))
sns.heatmap(
    xgb_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("XGBoost — Confusion Matrix")
plt.tight_layout()
plt.show()


## 5. Conclusion & caveats

Both boosters reach a **very high `balanced_accuracy` (~0.995+)**. On a **synthetic / engineered** dataset, such a score is plausible if the target is generated from strong rules already present in the features (server age, UPS health, efficiency, load, etc.).

Keep in mind:
- a near-perfect score does not automatically mean a "magic" model for real-world failures in production;
- `energy_efficiency_class` remains among the features: it can be informative without being an obvious leak (observed High↔Medium / High↔Low confusions still look coherent), but a feature-importance / ablation check would help isolate its contribution;
- a useful next step would be **feature importance** analysis and, optionally, a linear baseline to gauge the true difficulty of the task.

As it stands: clear experiment setup, `NA` parsing fixed, suitable metric, and LightGBM vs XGBoost comparison on a stratified hold-out.
